# Specificity of the MassQL queries against the GNPS libraries

Supports *Technical validation – Specificity of the queries*. Runs each query against the combined GNPS public libraries (MGF) locally and saves the hits per query; hits are then classified as steroidal (true) or not (false) to estimate the FDR. Shown here for the C24 queries; the same procedure applies to every query in Supplementary Table 1.

**Input:** a single MGF with the GNPS public libraries (`input_mgf` in the Settings cell).

In [ ]:
import os

# Root of the local project folder (the one holding Falcon/, Queries_final_mgf/,
# final_mgfs_library_generation/, ...). Edit this single line to run elsewhere.
PROJECT_DIR = '/media/dorrestein/Helena2/Bile_acids_update'

# MassQL queries on an MGF file

Fill in the **Settings** cell, then run all cells. Every query in `queries` runs one after the other on the same MGF.
Use the `general` kernel (massql is installed there).

In [ ]:
import os
from datetime import datetime

import numpy as np
import pandas as pd
from matchms.importing import load_from_mgf
from massql import msql_engine
from tqdm import tqdm

### MGF loader with a progress bar
This replaces massql's own MGF reader, which has two problems:
- It reads `metadata["pepmass"]` and `metadata["rtinseconds"]`, which current matchms renames, so every precursor m/z and RT
  comes out as 0. That breaks `MS2PREC` and `RTMIN`/`RTMAX`.
- It builds one Python dict per peak, which is slow and memory-hungry for large library files.

This version builds column arrays instead, shows a progress bar, and also returns a per-spectrum table
(spectrum ID, compound name, etc.) so hits can be traced back to library entries.

In [ ]:
def count_spectra(path, chunk_size=64 * 1024 * 1024):
    """Count spectra quickly by scanning raw bytes for BEGIN IONS."""
    n = 0
    tail = b""
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            data = tail + chunk
            n += data.count(b"BEGIN IONS")
            # keep a short overlap so a marker split across chunks is not missed (or double counted)
            tail = data[-9:] if not data.endswith(b"BEGIN IONS") else b""
    return n


def _first(value):
    """matchms returns pepmass as a tuple and charge as a list when not harmonized."""
    if isinstance(value, (list, tuple, np.ndarray)):
        return value[0] if len(value) else None
    return value


# Output column -> metadata keys to try. matchms renames some MGF keys even without harmonization
# (SPECTRUMID -> spectrum_id, NAME -> compound_name), so both spellings are listed.
META_FIELDS = {
    "spectrumid": ("spectrum_id", "spectrumid"),
    "name": ("compound_name", "name"),
    "libraryname": ("libraryname",),
    "smiles": ("smiles",),
    "ionmode": ("ionmode",),
}


def load_mgf_with_progress(path, meta_fields=META_FIELDS):
    n_total = count_spectra(path)

    mz_parts, i_parts, i_norm_parts, i_tic_parts = [], [], [], []
    scans, rts, precmzs, charges, n_peaks = [], [], [], [], []
    meta_rows = []

    spectra = load_from_mgf(path, metadata_harmonization=False)
    for spec_idx, spectrum in enumerate(tqdm(spectra, total=n_total, desc="Loading MGF", unit=" spectra", mininterval=1)):
        mz = spectrum.peaks.mz
        inten = spectrum.peaks.intensities
        keep = inten > 0
        if not keep.any():
            continue
        mz, inten = mz[keep], inten[keep]

        get = spectrum.get
        try:
            scan = int(get("scans"))
        except (TypeError, ValueError):
            scan = spec_idx + 1
        rt = get("retention_time", get("rtinseconds"))  # seconds
        precmz = _first(get("pepmass", get("precursor_mz")))
        charge = _first(get("charge"))

        mz_parts.append(mz)
        i_parts.append(inten)
        i_norm_parts.append(inten / inten.max())
        i_tic_parts.append(inten / inten.sum())
        scans.append(scan)
        rts.append(float(rt) / 60 if rt is not None else 0.0)
        precmzs.append(float(precmz) if precmz is not None else 0.0)
        charges.append(int(charge) if charge else 1)
        n_peaks.append(len(mz))

        row = {"scan": scan}
        for column, keys in meta_fields.items():
            row[column] = next((get(k) for k in keys if get(k) is not None), None)
        meta_rows.append(row)

    n_peaks = np.array(n_peaks)
    ms2_df = pd.DataFrame({
        "i": np.concatenate(i_parts),
        "i_norm": np.concatenate(i_norm_parts),
        "i_tic_norm": np.concatenate(i_tic_parts),
        "mz": np.concatenate(mz_parts),
        "scan": np.repeat(scans, n_peaks),
        "rt": np.repeat(rts, n_peaks),
        "precmz": np.repeat(precmzs, n_peaks),
        "ms1scan": 0,
        "charge": np.repeat(charges, n_peaks),
        "polarity": 1,
    })
    # Same placeholder MS1 table massql's own MGF loader returns (MGF files have no MS1 data)
    ms1_df = ms2_df.iloc[[-1]].reset_index(drop=True)
    spectra_meta_df = pd.DataFrame(meta_rows)
    return ms1_df, ms2_df, spectra_meta_df

## Settings

In [ ]:
# Path to the input MGF file
input_mgf = "/media/dorrestein/Helena2/GNPS_libraries_Oct2026/combined_libraries/Standard_combined.mgf"

# Folder containing the per-query output folders (each is created if it doesn't exist)
base_output_dir = f"{PROJECT_DIR}/FDR_estimation_MassQL_design_C24"

# Query name -> (output subfolder, MassQL query).
# Results are saved as <subfolder>/<query name>.tsv, plus <query name>_query.txt
queries = {
    "C24 - Nonhydroxy": (
        "C24_nonhydroxy_massQL_GNPS_libraries",
        "QUERY scaninfo(MS2DATA) WHERE MS2PROD=343.30:TOLERANCEMZ=0.01:INTENSITYPERCENT=5 AND MS2PROD=325.29:TOLERANCEMZ=0.01:INTENSITYPERCENT=5",
    ),
    "C24 - Monohydroxy": (
        "C24_monohydroxy_massQL_GNPS_libraries",
        "QUERY scaninfo(MS2DATA) WHERE MS2PROD=341.28:TOLERANCEMZ=0.01:INTENSITYPERCENT=5 AND MS2PROD=323.27:TOLERANCEMZ=0.01:INTENSITYPERCENT=5",
    ),
    "C24 - Dihydroxy": (
        "C24_dihydroxy_massQL_GNPS_libraries",
        "QUERY scaninfo(MS2DATA) WHERE MS2PROD=339.27:TOLERANCEMZ=0.01:INTENSITYPERCENT=5 AND MS2PROD=321.26:TOLERANCEMZ=0.01:INTENSITYPERCENT=5",
    ),
    "C24 - Trihydroxy": (
        "C24_trihydroxy_massQL_GNPS_libraries",
        "QUERY scaninfo(MS2DATA) WHERE MS2PROD=337.25:TOLERANCEMZ=0.01:INTENSITYPERCENT=5 AND MS2PROD=319.24:TOLERANCEMZ=0.01:INTENSITYPERCENT=5",
    ),
    "C24 - Tetrahydroxy": (
        "C24_tetrahydroxy_massQL_GNPS_libraries",
        "QUERY scaninfo(MS2DATA) WHERE MS2PROD=335.24:TOLERANCEMZ=0.01:INTENSITYPERCENT=5 AND MS2PROD=317.23:TOLERANCEMZ=0.01:INTENSITYPERCENT=5",
    ),
    "C24 - Pentahydroxy": (
        "C24_pentahydroxy_massQL_GNPS_libraries",
        "QUERY scaninfo(MS2DATA) WHERE MS2PROD=333.22:TOLERANCEMZ=0.01:INTENSITYPERCENT=5 AND MS2PROD=315.21:TOLERANCEMZ=0.01:INTENSITYPERCENT=5",
    ),
}

## Load the MGF

In [ ]:
if not os.path.isfile(input_mgf):
    raise FileNotFoundError(f"Input file not found: {input_mgf}")

# Loading is the slow part, so the data stay in memory: rerunning with only new queries skips it.
# (Restart the kernel if the MGF file itself changes but keeps the same path.)
if globals().get("_loaded_mgf") != input_mgf:
    ms1_df, ms2_df, spectra_meta_df = load_mgf_with_progress(input_mgf)
    _loaded_mgf = input_mgf
    print(f"Loaded {spectra_meta_df.shape[0]:,} spectra ({len(ms2_df):,} peaks).")
else:
    print("Using MGF data already loaded in memory.")

## Run all queries and save results

In [ ]:
summary_rows = []
all_results = {}

for query_name, (subfolder, query) in tqdm(queries.items(), desc="Queries", unit=" query"):
    output_dir = os.path.join(base_output_dir, subfolder)
    os.makedirs(output_dir, exist_ok=True)
    query_clean = " ".join(query.split())

    results_df = msql_engine.process_query(query_clean, input_mgf, ms1_df=ms1_df, ms2_df=ms2_df)
    if "scan" in results_df.columns:
        results_df = results_df.merge(spectra_meta_df, on="scan", how="left")
    else:  # massql returns a column-less table when nothing matches
        results_df = pd.DataFrame(columns=spectra_meta_df.columns)
    all_results[query_name] = results_df

    results_path = os.path.join(output_dir, query_name + ".tsv")
    results_df.to_csv(results_path, sep="\t", index=False)

    # Keep the query alongside the results for reproducibility
    with open(os.path.join(output_dir, query_name + "_query.txt"), "w") as f:
        f.write(f"name: {query_name}\n")
        f.write(f"input: {os.path.abspath(input_mgf)}\n")
        f.write(f"date: {datetime.now().isoformat(timespec='seconds')}\n")
        f.write(f"query: {query_clean}\n")
        f.write(f"n_rows: {len(results_df)}\n")

    summary_rows.append({"query": query_name, "rows": len(results_df),
                         "unique_scans": results_df["scan"].nunique(), "saved_to": results_path})

summary_df = pd.DataFrame(summary_rows)
summary_df